# Paired Seed-Level Statistical Analysis: Curriculum C vs. Flat C

## Overview
This notebook performs paired statistical significance testing and confidence interval estimation comparing:
- **Curriculum PPO Config C (v1)**: Phase 3 full refinement outputs across seeds 42, 123, and 777.
- **Flat PPO Config C (v1)**: Matched-compute flat training outputs across seeds 42, 123, and 777.

## Methodology
- **Design**: Fully paired by training seed ($n = 3$), evaluating difference $\Delta = \text{Curriculum C} - \text{Flat C}$.
- **Inference**: Student-$t$ distribution with $df = n - 1 = 2$ at 95% confidence level ($t_{0.975, df=2} = 4.3027$).
- **Evaluation Target**: Toxic-only test subset, analyzed both overall and stratified by difficulty bands (Easy, Medium, Hard).
- **Target Metrics**: Style Transfer Accuracy (STA), Semantic Similarity (Sim), and Severe Length Reduction rate (SLR $< 0.70$).
- **Data Integrity**: Purely data-driven from evaluation CSVs; no model training, no live inference, and zero hardcoded metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import Counter
from scipy.stats import t
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/Data_Khoaluan')
SOURCE_DIR = {
    'Curriculum C': ROOT/'ppo_config_c_v1'/'final_results',
    'Flat C': ROOT/'flat_ppo_config_c_v1'/'final_results',
}
SOURCE_SUFFIX = {'Curriculum C':'phase3_refinement', 'Flat C':'phase_flat'}
OUT = ROOT/'N4_C_vs_FlatC_Paired_Statistics'
SEEDS = [42, 123, 777]  # Three experimental seeds supplied by the study
BANDS = ['Easy','Medium','Hard','Overall']
METRICS = ['STA','Sim','SLR']
SLR_LIMIT = 0.70  # Confirmed original definition: strict less-than
CONFIDENCE = 0.95 # Requested in N4
assert len(SEEDS) >= 2 and len(SEEDS) == len(set(SEEDS))
assert len(SOURCE_DIR)==2
print('Inputs:', SOURCE_DIR)
print('Seeds:', SEEDS)

Mounted at /content/drive
Inputs: {'Curriculum C': PosixPath('/content/drive/MyDrive/Data_Khoaluan/ppo_config_c_v1/final_results'), 'Flat C': PosixPath('/content/drive/MyDrive/Data_Khoaluan/flat_ppo_config_c_v1_chaylai/final_results')}
Seeds: [42, 123, 777]


## 1. Load original six CSVs, validate schema, and align the evaluated input sets

The order of rows is **not** assumed to be identical. First verify each file has the same multiset of `(X, normalized label, normalized band)`, including duplicate counts. If any input/label/band combination is missing or duplicated inconsistently, stop. For the seed-level mean comparisons, row-level reordering cannot change the means; the verification ensures both experiments evaluated the same input set.

In [ ]:
def csv_path(method, seed):
    return SOURCE_DIR[method] / f'per_sample_seed{seed}_{SOURCE_SUFFIX[method]}.csv'

def read_eval(method, seed):
    path = csv_path(method, seed)
    if not path.is_file():
        raise FileNotFoundError(f'Missing source evaluation CSV: {path}')
    d = pd.read_csv(path, keep_default_na=False)
    required = {'X','Y','label','band','STA','Sim'}
    missing = required - set(d.columns)
    if missing:
        raise ValueError(f'{path}: missing columns {sorted(missing)}')
    for col in ['X','Y']:
        d[col] = d[col].astype(str)
        if d[col].str.strip().eq('').any():
            raise ValueError(f'{path}: blank {col} values present')
    d['_label'] = d['label'].astype(str).str.strip().str.lower()
    d['_band']  = d['band'].astype(str).str.strip().str.lower()
    if d['_label'].eq('').any() or d['_band'].eq('').any():
        raise ValueError(f'{path}: blank label or band')
    for metric in ['STA','Sim']:
        d[metric] = pd.to_numeric(d[metric], errors='raise')
        if not np.isfinite(d[metric].to_numpy()).all():
            raise ValueError(f'{path}: nonfinite {metric}')
    x_tokens = d['X'].str.split().map(len).clip(lower=1)
    y_tokens = d['Y'].str.split().map(len)
    d['SLR'] = ((y_tokens/x_tokens)<SLR_LIMIT).astype(float)
    return d

data = {(method,seed):read_eval(method,seed)
        for method in SOURCE_DIR for seed in SEEDS}

def sample_signature(d):
    return Counter(zip(d['X'].tolist(),d['_label'].tolist(),d['_band'].tolist()))

reference = data['Curriculum C',SEEDS[0]]
reference_signature = sample_signature(reference)
for (method,seed), d in data.items():
    sig=sample_signature(d)
    if sig != reference_signature:
        missing = list((reference_signature-sig).items())[:3]
        extra = list((sig-reference_signature).items())[:3]
        raise ValueError(
            f'Sample-set mismatch: {method}, seed {seed}. '
            f'Missing examples: {missing}; extra examples: {extra}'
        )
print(f'PASS: All {len(data)} CSV files evaluate the SAME input multiset.')
print('Observed total rows per CSV:',len(reference))
print('Observed toxic-only rows per CSV:',int(reference['_label'].eq('toxic').sum()))
print('Observed labels:',reference['_label'].value_counts().to_dict())
print('Observed bands:',reference['_band'].value_counts().to_dict())

PASS: All 6 CSV files evaluate the SAME input multiset.
Observed total rows per CSV: 1312
Observed toxic-only rows per CSV: 1150
Observed labels: {'toxic': 1150, 'nontoxic': 162}
Observed bands: {'easy': 636, 'medium': 379, 'hard': 297}


## 2. Paired seed-level differences and 95% Student-t CI

For each band and metric, first compute **one mean per seed and method** from toxic-only outputs, then subtract the matched seed's Flat C mean from its Config C Phase 3 mean. This yields `n=len(SEEDS)` paired differences. A sign check is descriptive and not itself a significance test.

No raw number of test rows is used as the number of independent training runs.

In [ ]:
def toxic_subset(d, band):
    out = d.loc[d['_label'].eq('toxic')]
    if band != 'Overall':
        out = out.loc[out['_band'].eq(band.lower())]
    if out.empty:
        raise ValueError(f'No toxic rows available for band {band}')
    return out

seed_rows=[]
delta_rows=[]
for band in BANDS:
    for metric in METRICS:
        deltas=[]
        for seed in SEEDS:
            curr=toxic_subset(data['Curriculum C',seed],band)
            flat=toxic_subset(data['Flat C',seed],band)
            if len(curr)!=len(flat):
                raise ValueError(f'{band}, seed {seed}: toxic subset sample counts differ')
            a=float(curr[metric].mean())
            b=float(flat[metric].mean())
            diff=a-b
            deltas.append(diff)
            seed_rows.extend([
                dict(Band=band, Metric=metric, Seed=seed, Method='Curriculum C', N=len(curr), Mean=a),
                dict(Band=band, Metric=metric, Seed=seed, Method='Flat C', N=len(flat), Mean=b),
            ])
        x=np.asarray(deltas,dtype=float)
        n=len(x)
        df=n-1
        tcrit=float(t.ppf((1+CONFIDENCE)/2,df))
        mean=float(x.mean())
        sd=float(x.std(ddof=1))
        half_width=tcrit*sd/np.sqrt(n)
        low,high=mean-half_width,mean+half_width
        row=dict(Band=band,Metric=metric,N_seeds=n,DF=df,
                 Delta_mean=mean,SD_paired_delta=sd,
                 CI95_low=low,CI95_high=high,
                 All_seeds_same_nonzero_sign=bool(np.all(x>0) or np.all(x<0)),
                 CI_excludes_zero=bool(low>0 or high<0),
                 Delta_percentage_points=100*mean if metric=='SLR' else np.nan)
        row.update({f'Delta_seed_{seed}':float(value) for seed,value in zip(SEEDS,x)})
        delta_rows.append(row)

per_seed=pd.DataFrame(seed_rows)
paired=pd.DataFrame(delta_rows)
display(paired.round(5))

,Band,Metric,N_seeds,DF,Delta_mean,SD_paired_delta,CI95_low,CI95_high,All_seeds_same_nonzero_sign,CI_excludes_zero,Delta_percentage_points,Delta_seed_42,Delta_seed_123,Delta_seed_777
0,Easy,STA,3,2,0.00191,0.00829,-0.01869,0.02251,False,False,NaN,-0.00092,0.01125,-0.00459
1,Easy,Sim,3,2,-0.02664,0.00754,-0.04537,-0.00791,True,True,NaN,-0.03533,-0.02273,-0.02185
2,Easy,SLR,3,2,0.11133,0.05509,-0.02553,0.24818,True,False,11.13256,0.17375,0.09073,0.06950
3,Medium,STA,3,2,0.01989,0.02743,-0.04826,0.08804,False,False,NaN,0.02339,0.04540,-0.00913
4,Medium,Sim,3,2,-0.02569,0.00778,-0.04502,-0.00637,True,True,NaN,-0.03368,-0.02526,-0.01814
5,Medium,SLR,3,2,0.11706,0.06763,-0.05093,0.28506,True,False,11.70635,0.17857,0.12798,0.04464
6,Hard,STA,3,2,0.04306,0.03514,-0.04423,0.13035,True,False,NaN,0.07132,0.05413,0.00371
7,Hard,Sim,3,2,-0.03337,0.01239,-0.06414,-0.00260,True,True,NaN,-0.04577,-0.03334,-0.02099
8,Hard,SLR,3,2,0.06644,0.09303,-0.16467,0.29755,False,False,6.64414,0.11149,0.12838,-0.04054
9,Overall,STA,3,2,0.01776,0.01902,-0.02950,0.06501,False,False,NaN,0.02478,0.03227,-0.00378


## 3. Per-method mean ± SD and descriptive checks

The published mean and SD should be obtained from these actual CSVs. **No numbers from the paper are hardcoded into the computation.** The report uses sample SD across seeds (`ddof=1`).

In [ ]:
summary=(per_seed.groupby(['Band','Metric','Method'],sort=False)
          .agg(Mean=('Mean','mean'),SD=('Mean','std'),N_seeds=('Seed','nunique'))
          .reset_index())
display(summary.round(5))
for _,r in paired.iterrows():
    v=[r[f'Delta_seed_{s}'] for s in SEEDS]
    print(f"{r['Band']:7s} {r['Metric']:3s} | "
          f"Δ={r['Delta_mean']:+.5f}, CI95=[{r['CI95_low']:+.5f}, {r['CI95_high']:+.5f}] "
          f"| seed deltas={dict(zip(SEEDS,np.round(v,5)))} | same sign={r['All_seeds_same_nonzero_sign']}")

,Band,Metric,Method,Mean,SD,N_seeds
0,Easy,STA,Curriculum C,0.91752,0.00693,3
1,Easy,STA,Flat C,0.91561,0.00155,3
2,Easy,Sim,Curriculum C,0.90181,0.00556,3
3,Easy,Sim,Flat C,0.92845,0.00750,3
4,Easy,SLR,Curriculum C,0.84170,0.03065,3
5,Easy,SLR,Flat C,0.73037,0.04769,3
6,Medium,STA,Curriculum C,0.91365,0.02419,3
7,Medium,STA,Flat C,0.89376,0.00478,3
8,Medium,Sim,Curriculum C,0.90675,0.00701,3
9,Medium,Sim,Flat C,0.93244,0.00656,3


Easy    STA | Δ=+0.00191, CI95=[-0.01869, +0.02251] | seed deltas={42: np.float64(-0.00092), 123: np.float64(0.01125), 777: np.float64(-0.00459)} | same sign=False
Easy    Sim | Δ=-0.02664, CI95=[-0.04537, -0.00791] | seed deltas={42: np.float64(-0.03533), 123: np.float64(-0.02273), 777: np.float64(-0.02185)} | same sign=True
Easy    SLR | Δ=+0.11133, CI95=[-0.02553, +0.24818] | seed deltas={42: np.float64(0.17375), 123: np.float64(0.09073), 777: np.float64(0.0695)} | same sign=True
Medium  STA | Δ=+0.01989, CI95=[-0.04826, +0.08804] | seed deltas={42: np.float64(0.02339), 123: np.float64(0.0454), 777: np.float64(-0.00913)} | same sign=False
Medium  Sim | Δ=-0.02569, CI95=[-0.04502, -0.00637] | seed deltas={42: np.float64(-0.03368), 123: np.float64(-0.02526), 777: np.float64(-0.01814)} | same sign=True
Medium  SLR | Δ=+0.11706, CI95=[-0.05093, +0.28506] | seed deltas={42: np.float64(0.17857), 123: np.float64(0.12798), 777: np.float64(0.04464)} | same sign=True
Hard    STA | Δ=+0.04306,

## 4. Export outputs (new analysis folder only)

- `paired_deltas_95CI.csv`: 12 comparisons (4 bands × 3 metrics), including per-seed differences.
- `per_seed_metrics.csv`: method × seed × band × metric.
- `means_sd_by_band.csv`: reconstructed means and standard deviations.
- `paired_CI_latex.tex`: LaTeX companion table.
- `RUN_NOTES.txt`: methodology and sources.

**Inference caution:** with three independent training seeds, `df=2`; CIs can be very wide. A CI including zero does not establish equivalence. The sign check does not replace inference. These original-C vs Flat-C results also differ in reward-weight scheduling; they do not isolate curriculum scheduling alone.

In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
paired.to_csv(OUT/'paired_deltas_95CI.csv',index=False)
per_seed.to_csv(OUT/'per_seed_metrics.csv',index=False)
summary.to_csv(OUT/'means_sd_by_band.csv',index=False)

latex=[r'\begin{tabular}{llccl}',r'\toprule',
       'Band & Metric & $\\overline{\\Delta}$ & 95\\% CI & Same sign (all seeds) '+r'\\',
       r'\midrule']
for _,r in paired.iterrows():
    interval=f"[{r['CI95_low']:+.4f}, {r['CI95_high']:+.4f}]"
    latex.append(
        f"{r['Band']} & {r['Metric']} & ${r['Delta_mean']:+.4f}$ & "
        f"${interval}$ & {'Yes' if r['All_seeds_same_nonzero_sign'] else 'No'} "+r'\\'
    )
latex.extend([r'\bottomrule',r'\end{tabular}'])
(OUT/'paired_CI_latex.tex').write_text('\n'.join(latex)+'\n',encoding='utf-8')
notes=[
    'N4: Original Config C Phase3 refinement vs old Flat C, before Post-Val.',
    f'Seeds: {SEEDS}; n={len(SEEDS)}; df={len(SEEDS)-1}.',
    f'Confidence level: {CONFIDENCE}; SLR if word-count(Y)/max(word-count(X),1) < {SLR_LIMIT}.',
    'Toxic-only. Per-seed band means first; differences paired by same seed.',
    'All input multisets verified using X, normalized label, normalized band.',
    'No hardcoded row counts or paper means.',
    'LIMITATION: only three training seeds; reward schedules also differ.',
]
notes.extend(f'{k}: {v}' for k,v in SOURCE_DIR.items())
(OUT/'RUN_NOTES.txt').write_text('\n'.join(notes)+'\n',encoding='utf-8')
print('Saved analysis files:',OUT)
print('LaTeX table:')
print((OUT/'paired_CI_latex.tex').read_text())

Saved analysis files: /content/drive/MyDrive/Data_Khoaluan/N4_C_vs_FlatC_Paired_Statistics
LaTeX table:
\begin{tabular}{llccl}
\toprule
Band & Metric & $\overline{\Delta}$ & 95\% CI & Same sign (all seeds) \\
\midrule
Easy & STA & $+0.0019$ & $[-0.0187, +0.0225]$ & No \\
Easy & Sim & $-0.0266$ & $[-0.0454, -0.0079]$ & Yes \\
Easy & SLR & $+0.1113$ & $[-0.0255, +0.2482]$ & Yes \\
Medium & STA & $+0.0199$ & $[-0.0483, +0.0880]$ & No \\
Medium & Sim & $-0.0257$ & $[-0.0450, -0.0064]$ & Yes \\
Medium & SLR & $+0.1171$ & $[-0.0509, +0.2851]$ & Yes \\
Hard & STA & $+0.0431$ & $[-0.0442, +0.1303]$ & Yes \\
Hard & Sim & $-0.0334$ & $[-0.0641, -0.0026]$ & Yes \\
Hard & SLR & $+0.0664$ & $[-0.1647, +0.2975]$ & No \\
Overall & STA & $+0.0178$ & $[-0.0295, +0.0650]$ & No \\
Overall & Sim & $-0.0281$ & $[-0.0496, -0.0066]$ & Yes \\
Overall & SLR & $+0.1014$ & $[-0.0555, +0.2584]$ & Yes \\
\bottomrule
\end{tabular}

